In [7]:
import sys
import os
import pandas as pd
import mlflow.xgboost
from scripts.processing import extract_features  # Your module

# 1. CONSTANTS (Configurations)
MLRUNS_DIR = "/Users/nicholus/Documents/GitHub/credit-card-default/notebooks/mlruns"
MODEL_URI = "models:/CreditDefault_XGB/3"

def run_prediction(raw_data: dict, model):
    """
    The 'Worker' function. Takes raw data and a pre-loaded model.
    """
    df_raw = pd.DataFrame([raw_data])
    df_featured = extract_features(df_raw)
    
    # Ensure columns match the model's expected input
    X = df_featured.drop(columns=['ID', 'default'], errors='ignore').astype(float)
    
    prob = model.predict_proba(X)[:, 1][0]
    return {"probability": round(float(prob), 4), "label": int(prob > 0.5)}

def main():
    """
    Set up the environment and runs the test.
    """
    print("Initializing Inference Engine...")
    mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")
    
    # Load model once
    print(f" Loading model from {MODEL_URI}...")
    model = mlflow.xgboost.load_model(MODEL_URI)
    
    # Define a test customer
    test_customer = {
        "LIMIT_BAL": 120000, "SEX": 2, "EDUCATION": 2, "MARRIAGE": 2, "AGE": 26,
        "PAY_0": -1, "PAY_2": 2, "PAY_3": 0, "PAY_4": 0, "PAY_5": 0, "PAY_6": 2,
        "BILL_AMT1": 2682, "BILL_AMT2": 1725, "BILL_AMT3": 2682, 
        "BILL_AMT4": 3272, "BILL_AMT5": 3455, "BILL_AMT6": 3261,
        "PAY_AMT1": 0, "PAY_AMT2": 1000, "PAY_AMT3": 1000, 
        "PAY_AMT4": 1000, "PAY_AMT5": 0, "PAY_AMT6": 2000
    }
    
    # Run inference
    result = run_prediction(test_customer, model)
    
    print("-" * 30)
    print(f" Prediction Result: {result}")
    print("-" * 30)

if __name__ == "__main__":
    main()

Initializing Inference Engine...
 Loading model from models:/CreditDefault_XGB/3...
------------------------------
 Prediction Result: {'probability': 0.544, 'label': 1}
------------------------------
